# Gua jadiin satu disini ya


In [403]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain.tools import Tool
from dotenv import load_dotenv
import os

class FashionSearchAgent:
    def __init__(self):
        # Load environment variables
        load_dotenv()
        self.prompt_template = """This tool only used for searching product where the input is event or occasion. Your task is to search for real, available product links for fashion items based on the user's event or occasion input.

            IMPORTANT SEARCH INSTRUCTIONS:
            1. For each category, first formulate a specific search query like "baju pria formal lebaran tokopedia" or "sepatu wanita casual shopee"
            2. YOU MUST RETURN DIRECT PRODUCT LINKS ONLY. Valid links should point to specific products
            3. DO NOT use "find" or "search" keywords links like "https://www.tokopedia.com/find/..." or "https://shopee.co.id/search?keyword=..." - these are NOT acceptable
            4. Click through to actual product pages to get the direct product URLs
            5. If you cannot find a suitable item after searching, remove that category from the JSON object. Do not include empty categories.
            6. You need to make sure the link is available, because i always found that the link is not available anymore
            7. Only use links that point directly to specific product pages, never to search results pages.
            8. Use the following categories: "Top", "Bottom", "Outer", "Footwear", "Accessories", "Bag", "Dress", "Jumpsuit", "Swimwear", "Lingerie", "Sleepwear", "Activewear".
            9. Adjust to gender in request input

            Generate a JSON object following this exact structure:
            {
                "products": [
                    {
                        "category": "XXX",
                        "name": "<specific product name>",
                        "links": [
                            "<direct product link - NOT search results>",
                            "<direct product link - NOT search results>"
                        ]
                    },
                    {
                        "category": "XXX",
                        "name": "<specific product name>",
                        "links": [
                            "<direct product link - NOT search results>",
                            "<direct product link - NOT search results>"
                        ]
                    }
                ]
            }

            DO NOT MAKE UP LINKS. Only use links that point directly to specific product pages, never to search results pages.
            """
        # Initialize the LLM
        self.llm = ChatGroq(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            temperature=0,
            seed=42,
            top_p=0.002
        )

        # Initialize the search tool
        self.search_tool = Tool(
            name="Product Search based on event",
            func=TavilySearchResults(
                api_key=os.getenv("TAVILY_API_KEY"),
                max_results=10,
                search_depth='basic',
                topic="fashion",
                include_domains=["https://shop-id.tokopedia.com/"],
                exclude_domains=[
                    "https://www.tokopedia.com/find/",
                    "https://shopee.co.id/search?keyword="
                ]
            ),
            description= self.prompt_template,
            max_retries=5,
            verbose=True,
        )

        # Create the agent
        self.agent_executor = create_react_agent(self.llm, [self.search_tool])

    def search(self, user_input: str) -> str:
        response = self.agent_executor.invoke({"messages": user_input})
        return response['messages'][-1].content

In [404]:
import base64
import os
from dotenv import load_dotenv
from groq import Groq

class OutfitDescriber:
    def __init__(self, model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, top_p=0.002, seed=42):
        load_dotenv()
        self.client = Groq()
        self.model = model
        self.temperature = temperature
        self.top_p = top_p
        self.seed = seed

    def encode_image(self, image_path):
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')

    def describe_outfit(self, image_path: str) -> str:
        b64_image = self.encode_image(image_path)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "describe the outfit in the photo, only the outfit, no other details",
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64_image}",
                            }
                        }
                    ]
                }
            ],
            temperature=self.temperature,
            top_p=self.top_p,
            max_completion_tokens=1024,
            stream=False,
            seed=self.seed,
        )

        return response.choices[0].message.content

In [405]:
class FashionAppAgent:
    def __init__(self):
        self.fashion_search_agent = FashionSearchAgent()
        self.outfit_describer = OutfitDescriber()

    def check_input_image_or_event(self, user_input):
        if user_input.lower().endswith(('.jpg', '.jpeg', '.png')):
            return "image"
        else:
            return "event"

    def main(self, user_input):
        input_type = self.check_input_image_or_event(user_input)
        if input_type == "image":
            # First get description from image
            outfit_description = self.outfit_describer.describe_outfit(user_input)
            # Then search for products based on that description
            result = self.fashion_search_agent.search(outfit_description)
            return result
        else:
            # If it's text, just search directly
            result = self.fashion_search_agent.search(user_input)
            return result

Agent = FashionAppAgent()

c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! seed is not default parameter.
                    seed was transferred to model_kwargs.
                    Please confirm that seed is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  warnings.warn(


In [407]:
# Example usage

# input = r"D:\Work\Programming\Web Developer\Hacktiv8\Phase 3\Final Project\stylehack\ai\coba.jpg"
input = "I'm men, i'm going to a natal wedding and I need a dress for the event"
result = Agent.main(input)
print("\n\n", result)

content='[{"title": "Jas Pengantin Pria , Setelan Jas pengantin , Setelan jas wedding - Shop ...", "url": "https://shop-id.tokopedia.com/pdp/1729909131361684184", "content": "Buy Jas Pengantin Pria , Setelan Jas pengantin , Setelan jas wedding on Shop | Tokopedia. Discover great prices on Setelan Resmi and get free shipping on eligible items. Shop now for exclusive deals!", "score": 0.60861784}, {"title": "vest- formal rompi setelan dalaman jas pria / groomsmen wedding- - Shop ...", "url": "https://shop-id.tokopedia.com/pdp/1729622075343079660", "content": "Buy vest- formal rompi setelan dalaman jas pria / groomsmen wedding- on Shop | Tokopedia. Discover great prices on Rompi & Gilet and get free shipping on eligible items. Shop now for exclusive deals!", "score": 0.5849357}, {"title": "Jas Setelan Semiwool Wedding Pria Jas formal non formal Blazer - Shop ...", "url": "https://shop-id.tokopedia.com/pdp/1729445302104918351", "content": "Buy Jas Setelan Semiwool Wedding Pria Jas formal n